# LLM Inference Parameters Demo

A short live-coding notebook for AI Academy.

In this notebook, we will call an OpenAI model several times and observe how inference parameters change the output.

## 1. Setup

We use the official OpenAI Python SDK. The API key should be available as `OPENAI_API_KEY`.

For local demos, you can edit a `.env` file in the same folder where you launch Jupyter, and add the corresponding API_KEY.

In [ ]:
from utils.open_ai import OpenAI
from dotenv import load_dotenv
load_dotenv()
import os

# ---------------------------------------------------------------------------
# Same client / MODEL setup as before — no other cells need to change.
# ---------------------------------------------------------------------------

print("Probando")

client = OpenAI()
# Map notebook model names to the names registered in LiteLLM

def call_llm(
    prompt,
    system_prompt=None,
    temperature=0.7,
    top_p=1.0,
    max_output_tokens=120,
    seed=None,
):
    """Call the LiteLLM Responses-compatible API and print the model output."""
    messages = []
    if system_prompt:
        messages.append({"role": "system", "content": system_prompt})
    messages.append({"role": "user", "content": prompt})

    request = {
        "model": os.environ.get('MODEL'),
        "input": messages,
        "temperature": temperature,
        "top_p": top_p,
        "max_output_tokens": max_output_tokens,
    }

    if seed is not None:
        request["seed"] = seed

    optional_params = ["seed", "top_p", "temperature"]
    removed_params = []

    while True:
        try:
            response = client.responses.create(**request)
            break
        except TypeError as error:
            removed = False
            for param in optional_params:
                if param in request:
                    removed_params.append(param)
                    request.pop(param)
                    removed = True
                    break
            if not removed:
                raise error
        except Exception as error:
            message = str(error).lower()
            removed = False
            for param in optional_params:
                if param in request and param in message:
                    removed_params.append(param)
                    request.pop(param)
                    removed = True
                    break
            if not removed:
                raise error

    if removed_params:
        print("Note: Retried without unsupported parameter(s):", ", ".join(removed_params))

    print(response.output_text)
    return response




## 2. Baseline Call

First, we make one simple request using the default settings.

Prompt: `Write a short product description for a smartwatch.`

In [4]:
product_prompt = "Write a short product description for a smartwatch."

print("Baseline output")
print("-" * 60)
call_llm(product_prompt)

Baseline output
------------------------------------------------------------
Stay connected and track your health effortlessly with our sleek smartwatch. Featuring a vibrant touchscreen, heart rate monitoring, GPS, and customizable notifications, it’s the perfect companion for your active lifestyle. Durable, stylish, and compatible with both iOS and Android devices, this smartwatch keeps you motivated and on schedule wherever you go.


## 3. Temperature Demo

`temperature` controls randomness.

Lower values usually produce more predictable and focused output. Higher values generally increase variety and creativity.

In [5]:
for temp in [2, 2, 2]:
    print(f"Temperature = {temp}")
    print("-" * 60)
    call_llm(product_prompt, temperature=temp, max_output_tokens=100)
    print("\n")

Temperature = 2
------------------------------------------------------------
Stay connected and track your health effortlessly with our latest smartwatch. Featuring a sleek design, heart rate monitoring, sleep tracking, activity tracking, and customizable watch faces, it’s the perfect companion for your active lifestyle. Compatible with both iOS and Android, this smartwatch ensures you never miss a call, message, or notification. Arm yourself with convenience and style – right on your wrist.


Temperature = 2
------------------------------------------------------------
Stay connected and elevate your health with the sleek SportPro Smartwatch. Featuring a bright touchscreen, 24/7 heart rate monitoring, GPS tracking, and customizable watch faces, it keeps you motivated both in and out of the gym. Receive calls, texts, and app notifications directly on your wrist. Designed with comfort and durability in mind, the SportPro powers your active lifestyle effortlessly.


Temperature = 2
------

## 4. Top-p Demo

`top_p` controls how much of the probability distribution is considered when choosing the next token.

A smaller value, such as `0.3`, restricts choices to a smaller set of likely tokens. A value of `1.0` allows the model to consider the full distribution.

Cumulative Probability: 
The AI lists possible next words from most likely to least likely and adds their scores together.
The Threshold (p): You set a value for p between 0 and 1 (such as 0.90 for 90%).
Dynamic Pool: The model stops adding words as soon as the running total crosses your p value.
Adaptive Size: If the AI is very sure of the next word, the pool shrinks to just one or two options. If the AI is unsure, the pool grows to include more creative choices. 
Choosing the Right Value
Low Top-P (0.1–0.4): Produces focused, predictable, and factual answers. Best for math or coding.
Medium Top-P (0.5–0.8): Balances accuracy and creativity for everyday writing.
High Top-P (0.9–1.0): Expands the word pool for diverse and creative

In this example:
Here we keep `temperature` fixed at `0.8` and change only `top_p`.

Additional example: Top-k
Fixed limit: The AI looks at all possible next words, ranks them by probability, and keeps only the top K choices.Rebalancing: It ignores all other words and recalculates the probabilities for the remaining K options to pick one.Low K vs. High K: A low K (like 1 or 3) makes the output predictable, safe, and factual. A high K (like 50 or 100) lets the model consider more words, which increases creativity and variety

In [6]:
for top_p_value in [0.3, 1.0]:
    print(f"top_p = {top_p_value}")
    print("-" * 60)
    call_llm(product_prompt, temperature=0.8, top_p=top_p_value, max_output_tokens=100)
    print("\n")

top_p = 0.3
------------------------------------------------------------
Stay connected and track your health effortlessly with our sleek smartwatch. Featuring a vibrant touchscreen, heart rate monitoring, GPS, and customizable notifications, it’s the perfect companion for your active lifestyle. Durable, stylish, and compatible with both iOS and Android devices, this smartwatch keeps you motivated and in control throughout your day.


top_p = 1.0
------------------------------------------------------------
Stay connected and track your health effortlessly with our sleek smartwatch. Featuring a vibrant touchscreen, heart rate monitoring, sleep tracking, and customizable notifications, it’s the perfect companion for your active lifestyle. Durable, stylish, and compatible with both iOS and Android devices, this smartwatch keeps you motivated and in control all day long.




## 5. Max Output Tokens Demo

`max_output_tokens` limits the maximum length of the model response.

A small value can cut the answer short. A larger value gives the model more room to answer.

In [7]:
for token_limit in [30, 150]:
    print(f"max_output_tokens = {token_limit}")
    print("-" * 60)
    call_llm(product_prompt, temperature=0.7, max_output_tokens=token_limit)
    print("\n")

max_output_tokens = 30
------------------------------------------------------------
Stay connected and track your health effortlessly with our sleek smartwatch. Featuring a vibrant touchscreen, heart rate monitoring, GPS, and customizable notifications, it’s the


max_output_tokens = 150
------------------------------------------------------------
Stay connected and track your health effortlessly with our sleek smartwatch. Featuring a vibrant touchscreen, heart rate monitoring, GPS, and customizable notifications, it’s the perfect companion for your active lifestyle. Durable, stylish, and compatible with both iOS and Android devices, this smartwatch keeps you motivated and in control all day long.




## 6. Seed / Reproducibility Demo

The seed parameter in generative AI is an integer value used to initialize the random number generator, allowing you to control the model's randomness and achieve reproducible results.
The token chosen is the first token whose cumulative probability exceeds `seed`

`seed` can help make outputs more reproducible when using the same prompt and parameters.

However, exact determinism may depend on the API, model, and SDK version.

We run the same prompt twice with `seed = 42`, then once with `seed = 7`.

In [8]:
print("Run 1: seed = 42")
print("-" * 60)
call_llm(product_prompt, temperature=0.8, max_output_tokens=100, seed=42)

print("\nRun 2: seed = 42")
print("-" * 60)
call_llm(product_prompt, temperature=0.8, max_output_tokens=100, seed=42)

print("\nRun 3: seed = 7")
print("-" * 60)
call_llm(product_prompt, temperature=0.8, max_output_tokens=100, seed=7)

Run 1: seed = 42
------------------------------------------------------------
Stay connected and track your health effortlessly with our sleek smartwatch. Featuring a vibrant touchscreen, heart rate monitoring, sleep tracking, and customizable notifications, it's the perfect blend of style and functionality for your active lifestyle.

Run 2: seed = 42
------------------------------------------------------------
Stay connected and track your health effortlessly with our sleek smartwatch. Featuring a vibrant touchscreen, heart rate monitoring, sleep tracking, and customizable notifications, it's the perfect blend of style and functionality for your active lifestyle.

Run 3: seed = 7
------------------------------------------------------------
Stay connected and track your health effortlessly with our sleek smartwatch. Featuring a vibrant touchscreen, heart rate monitoring, step tracking, and customizable notifications, it’s the perfect companion for your active lifestyle. Durable, stylis

## 7. System Instructions Demo

System instructions set the role, context, and behavior style for the model.

We use the same user prompt twice, but change the system instruction.

In [9]:
api_prompt = "Explain what an API is."

architect_system = "You are a senior software architect. Explain concepts precisely and technically."
teacher_system = "You are a patient teacher explaining concepts to absolute beginners."

print("System instruction: senior software architect")
print("-" * 60)
call_llm(api_prompt, system_prompt=architect_system, temperature=0.7, max_output_tokens=150)

print("\nSystem instruction: patient beginner teacher")
print("-" * 60)
call_llm(api_prompt, system_prompt=teacher_system, temperature=0.7, max_output_tokens=150)

System instruction: senior software architect
------------------------------------------------------------
An API (Application Programming Interface) is a defined set of protocols, routines, and tools that enable different software components to communicate and interact with each other. It specifies the methods and data formats that applications can use to request and exchange information, abstracting underlying implementation details.

Technically, an API defines:

- **Interface**: The exposed endpoints, function signatures, or message formats that a consumer can invoke.
- **Contracts**: The expected inputs, outputs, data types, and error handling behaviors.
- **Protocols**: The communication mechanisms such as HTTP/HTTPS for RESTful APIs, or method calls in libraries.

APIs enable modularity, interoperability, and integration by allowing software systems—whether internal components, third-party services, or hardware

System instruction: patient beginner teacher
----------------------

## 8. Prompt Engineering Demo

A vague prompt gives the model very little guidance.

A structured prompt tells the model what format, focus, and style we want.

In [10]:
company_text = """
BrightCart, a fictional online retail company, introduced AI assistants for its customer support team.
The assistants help agents find order details faster, draft replies, and identify urgent complaints.
After three months, average response time decreased by 35%, but managers noticed that employees needed training
to review AI-generated replies carefully before sending them to customers.
"""

vague_prompt = f"Summarize this.\n\n{company_text}"

structured_prompt = f"""
Summarize the following text in 3 bullet points.
Focus on business impact.
Use plain English.

Text:
{company_text}
"""

print("Vague prompt")
print("-" * 60)
call_llm(vague_prompt, temperature=0.5, max_output_tokens=120)

print("\nStructured prompt")
print("-" * 60)
call_llm(structured_prompt, temperature=0.5, max_output_tokens=150)

Vague prompt
------------------------------------------------------------
BrightCart implemented AI assistants to support its customer service team, resulting in a 35% reduction in average response time after three months. However, managers observed that employees require training to properly review AI-generated responses before sending them to customers.

Structured prompt
------------------------------------------------------------
- AI assistants helped BrightCart’s support team respond to customers 35% faster, improving service speed.  
- The technology made it easier for agents to access order info and handle urgent issues quickly.  
- Managers found employees need training to properly check AI-generated replies, ensuring quality communication.


## 9. Structured Output Demo

LLMs can extract information and return it in a structured format such as JSON.

For production applications, always validate the JSON before using it in another system.

In [ ]:
customer_message = """
Hi, my name is Priya Raman. My smartwatch stopped syncing after yesterday's update.
I use it for work calls, so this is urgent. Can someone help me fix it today or replace it?
"""

json_prompt = f"""
Extract information from the customer message below.
Return only valid JSON with these fields:
- customer_name
- issue
- urgency
- requested_action

Customer message:
{customer_message}
"""

call_llm(json_prompt, temperature=0, max_output_tokens=150)

```json
{
  "customer_name": "Priya Raman",
  "issue": "Smartwatch stopped syncing after yesterday's update",
  "urgency": "urgent",
  "requested_action": "help fix it today or replace it"
}
```


## 10. Wrap-up

Key ideas from this demo:

- `temperature` controls randomness and creativity.
- `top_p` controls the candidate token pool considered by the model.
- `max_output_tokens` controls the maximum response length.
- `seed` can help reproducibility, but exact determinism may depend on the API and model.
- System instructions steer the model's role, tone, and behavior.
- Better prompts usually produce more useful outputs.